# Playground: filter your own images

Upload a photo (a cat works beautifully), pick a kernel, choose a backend, and run. Try
`edges` on a photo for the dramatic outline, or `blur` / `sharpen` with **colour output**
ticked to filter all three RGB channels. On the board, flip the backend and watch the
latency change. For a full-resolution photo prefer `software` or the HLS backends; the
MMIO-streamed `rtl` path is happiest on small images.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

from fpga_conv import get_backend, BACKEND_LABELS
from fpga_conv.bench import image_from_bytes

sample_image = make_sample_image(256)

_cache = {}
def backend(name):
    if name not in _cache:
        kw = HW_KW if name in ('rtl', 'hls_naive', 'hls_opt', 'hls_stream') else {}
        _cache[name] = get_backend(name, **kw)
    return _cache[name]

In [ ]:
import traceback
import ipywidgets as widgets
from IPython.display import display

backend_opts = [(BACKEND_LABELS.get(n, n), n) for n in available_backends()]
kernel_dd  = widgets.Dropdown(options=list(KERNELS), value='edges', description='kernel:')
backend_tb = widgets.ToggleButtons(options=backend_opts, description='backend:')
color_cb   = widgets.Checkbox(value=False, description='colour output (filter R,G,B)')
upload     = widgets.FileUpload(accept='image/*', multiple=False, description='upload image')
run_btn    = widgets.Button(description='Run convolution', button_style='primary')
out_area   = widgets.Output()


def current_image():
    # FileUpload.value differs across ipywidgets versions (PYNQ ships v7): v8 is a tuple
    # of dicts with 'name' at top level; v7 is a {filename: {'metadata': {...}, ...}}
    # mapping. Handle both, and accept content as memoryview or bytes.
    val = upload.value
    if val:
        item = val[0] if isinstance(val, (list, tuple)) else next(iter(val.values()))
        name = item.get('name') or item.get('metadata', {}).get('name', 'upload')
        return name, image_from_bytes(bytes(item['content']))
    return None, sample_image


def on_run(_):
    with out_area:
        out_area.clear_output(wait=True)
        try:
            fname, image = current_image()
            kernel = get_kernel(kernel_dd.value)
            name = backend_tb.value
            src = f'uploaded "{fname}"' if fname else 'sample image'
            print(f'running {kernel.name} on {src} ({image.shape[1]}x{image.shape[0]}) '
                  f'via {BACKEND_LABELS.get(name, name)} ...')
            if name == 'rtl' and max(image.shape[:2]) > 160:
                from PIL import Image
                scale = 160 / max(image.shape[:2])
                new = (max(1, int(image.shape[1] * scale)), max(1, int(image.shape[0] * scale)))
                image = np.array(Image.fromarray(image).resize(new))
                print(f'  (downscaled to {new[0]}x{new[1]} for the MMIO-streamed RTL path)')
            result, ms = backend(name).run(image, kernel, color=color_cb.value)
            shown_in = image if (color_cb.value and image.ndim == 3) else to_grayscale_u8(image)
            fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
            ax[0].imshow(shown_in, cmap=None if shown_in.ndim == 3 else 'gray')
            ax[0].set_title('input'); ax[0].axis('off')
            ax[1].imshow(result, cmap=None if result.ndim == 3 else 'gray')
            ax[1].set_title(f'{kernel.name}  ({BACKEND_LABELS.get(name, name)}, {ms:.2f} ms)')
            ax[1].axis('off')
            fig.tight_layout(); display(fig); plt.close(fig)
            print('kernel matrix (>> %d, mode %d):' % (kernel.shift, kernel.mode))
            print(kernel.matrix)
        except Exception:
            print('run failed:\n' + traceback.format_exc())


run_btn.on_click(on_run)
upload.observe(lambda change: on_run(None), names='value')
display(widgets.VBox([widgets.HBox([kernel_dd, backend_tb]),
                      widgets.HBox([color_cb, upload, run_btn]), out_area]))
on_run(None)